# TTSIDC: truncated Taylor series iterative downward continuation (Zhang et al., 2013)

Classical baseline, no training. For every patch the smoothing parameter σ is chosen with the C-norm
criterion and the number of iterations at the knee of the data-misfit curve, so the method uses no
ground truth. Runs on CPU.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward with the multiscale operator (40 steps of
20 m), the same input that the multiscale models receive.

In [2]:
from gravdc import MultiscaleOperator
from gravdc import classical

names, x = load_test_set(DATA_DIR)
A = MultiscaleOperator()  # 40 x 20 m = 800 m
y = A(x)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Inference

In [3]:
TTSIDC = dict(cell=400, z=800, expand=10, window=3, taylor_down=6, taylor_up=3)

recs = []
for name, f in zip(names, y[:, 0].numpy()):
    sigma = classical.select_sigma_cnorm(f, n=50, sigma_start=1.2, ratio=1.05, iterations=4, **TTSIDC)
    k = classical.select_iterations(f, sigma, max_iter=100, **TTSIDC)
    recs.append(classical.ttsidc(f, sigma=sigma, iterations=k, **TTSIDC))
    print(f"{name:12s} sigma = {sigma:.4f}   iterations = {k}")
x_hat = np.stack(recs)[:, None]

data14.mat   sigma = 1.3230   iterations = 10
data16.mat   sigma = 1.3230   iterations = 10
data18.mat   sigma = 1.3230   iterations = 10
data20.mat   sigma = 1.5315   iterations = 10
data22.mat   sigma = 1.3230   iterations = 10
data24.mat   sigma = 1.5315   iterations = 10
data26.mat   sigma = 1.3230   iterations = 10
data28.mat   sigma = 1.3230   iterations = 10
data30.mat   sigma = 1.3892   iterations = 10
data32.mat   sigma = 1.3230   iterations = 10


## Metrics

In [4]:
per_image = evaluate(x, x_hat)
compare("TTSIDC", summarize(per_image))
save_result("TTSIDC", per_image, names, RESULTS_DIR)

TTSIDC
  metric             this run              Table 3
  PSNR       36.8247 ± 2.5611     36.8247 ± 2.5611  ✓
  SSIM        0.9882 ± 0.0076      0.9882 ± 0.0076  ✓
  RE          0.0746 ± 0.0243      0.0746 ± 0.0243  ✓
  RMSE        0.0262 ± 0.0097      0.0262 ± 0.0097  ✓


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"TTSIDC": x_hat[k]}, title=names[k])